# 날씨데이터와 암모니아 센터데이터 병합 후 출력 코드

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import re

weather_data = pd.read_csv('/content/drive/MyDrive/nh3 예측 폴더/data/암모니아날씨데이터_24-07-29-13시.csv')
sensor_data = pd.read_csv('/content/drive/MyDrive/nh3 예측 폴더/data/암모니아데이터_24-07-29-13시.csv')

# 날씨 데이터의 날짜 및 시간 조합
weather_data['fcst_datetime'] = pd.to_datetime(
    weather_data['fcst_date'].astype(str) + weather_data['fcst_time'].astype(str).str.zfill(4),
    format='%Y%m%d%H%M'
)

# 센서 데이터의 시간을 분 단위로 정리
sensor_data['crt_dttm'] = pd.to_datetime(sensor_data['crt_dttm'])
sensor_data['crt_dttm'] = sensor_data['crt_dttm'].dt.floor('T')

# 시간별로 데이터 확장 (각 시간당 60분),
weather_expanded = weather_data.set_index('fcst_datetime').groupby('sn', as_index=False).resample('T').ffill().reset_index()

# 장치 매핑
sensor_data['weather_sn'] = sensor_data['device_sn'].map({
    'SP_KH_D01': 'KH_WM_01',
    'SP_KH_D02': 'KH_WM_01',
    'SP_SB_D01': 'SB_WM_01',
    'SP_SB_D02': 'SB_WM_01',
    'SP_SB_D03': 'SB_WM_01'
})

# 데이터 병합
merged_data = pd.merge(sensor_data, weather_expanded, left_on=['crt_dttm', 'weather_sn'], right_on=['fcst_datetime', 'sn'], how='inner')

# 함수 정의
def convert_pcp(value):
    if '강수없음' in value:
        return 0.0
    elif '미만' in value:
        num = re.findall(r'\d+', value)
        if num:
            return float(num[0]) - 0.5  # 1mm 미만을 0.5로 변환
        else:
            return 0.0
    else:
        # 소수점 포함 숫자 추출
        num = re.findall(r'\d+\.\d+|\d+', value)
        if num:
            return float(num[0])
        else:
            return 0.0

# apply 메서드를 사용하여 함수 적용
merged_data['pcp'] = merged_data['pcp'].apply(convert_pcp)

selected_columns = merged_data[['sn', 'device_sn','temp', 'humi', 'pres', 'nh3_cv','fcst_datetime','tmp','reh','pcp','wsd','sky']]

selected_columns.to_excel('/content/drive/MyDrive/selected_columns_0729min.xlsx', index=False)

# 날씨데이터를 통해 맨홀온도 예측 모델 생성 코드

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib


# 데이터 로드
data = pd.read_excel('/content/drive/MyDrive/nh3 예측 폴더/data/selected_columns_0729min.xlsx')
data['timestamp'] = pd.to_datetime(data['fcst_datetime'])
data['hour'] = data['timestamp'].dt.hour
data['minute'] = data['timestamp'].dt.minute
# device_sn에 따라 데이터 분리
device_groups = data.groupby('device_sn')

results = {}
comparison_dfs = {}

for device_sn, group in device_groups:
    print(f"Processing device: {device_sn}")
    # 특성과 타겟 분리
    features = group[['tmp', 'reh', 'pcp', 'wsd', 'sky','hour','minute']]
    targets = group[['temp', 'humi', 'nh3_cv']]

    # 훈련 세트와 테스트 세트 분할
    X_train, X_test, y_train, y_test = train_test_split(features, targets, test_size=0.2, random_state=42)

    # 모델 훈련
    model_temp = RandomForestRegressor(max_depth=20, min_samples_split=5, n_estimators=400, random_state=42)
    model_temp.fit(X_train, y_train['temp'])

    model_humi = RandomForestRegressor(max_depth=20, min_samples_split=5, n_estimators=400, random_state=42)
    model_humi.fit(X_train, y_train['humi'])

    # 예측
    predicted_temp = model_temp.predict(X_test)
    predicted_humi = model_humi.predict(X_test)

    # 성능 평가
    temp_mse = mean_squared_error(y_test['temp'], predicted_temp)
    temp_r2 = r2_score(y_test['temp'], predicted_temp)
    humi_mse = mean_squared_error(y_test['humi'], predicted_humi)
    humi_r2 = r2_score(y_test['humi'], predicted_humi)

    # 결과 저장
    results[device_sn] = {
        'Temperature MSE': temp_mse,
        'Temperature R2': temp_r2,
        'Humidity MSE': humi_mse,
        'Humidity R2': humi_r2
    }

    # 모델 저장 (예시, 경로는 실제 상황에 맞게 조정)
    joblib.dump(model_temp, f'/content/drive/MyDrive/nh3 예측 폴더/model_temp_0729_min_{device_sn}.pkl')
    joblib.dump(model_humi, f'/content/drive/MyDrive/nh3 예측 폴더/model_humi_0729_min_{device_sn}.pkl')

     # 예측 결과와 실제 데이터를 데이터프레임으로 정리
    comparison_df = pd.DataFrame({
        'Feature_Temp': X_test['tmp'],
        'Feature_Reh': X_test['reh'],
        'Feature_Pcp': X_test['pcp'],
        'Feature_Wsd': X_test['wsd'],
        'Feature_Sky': X_test['sky'],
        'Feature_Hour': X_test['hour'],
        'Feature_Minute': X_test['minute'],
        'Actual_Temperature': y_test['temp'],
        'Predicted_Temperature': predicted_temp,
        'Actual_Humidity': y_test['humi'],
        'Predicted_Humidity': predicted_humi
    })

     # 딕셔너리에 comparison_df 저장
    comparison_dfs[device_sn] = comparison_df

      # 데이터프레임 출력 (여기서는 .head()를 사용하여 상위 5개 행만 출력)
    print(f"Comparison Results for Device {device_sn}")
    print(comparison_df.head())


    # 예측 결과와 실제 데이터를 준비
    actual_temp = y_test['temp']  # 실제 온도 데이터
    predicted_temp = model_temp.predict(X_test)  # 예측된 온도 데이터
    actual_humi = y_test['humi']  # 실제 습도 데이터
    predicted_humi = model_humi.predict(X_test)  # 예측된 습도 데이터

    # 그래프를 그리기 위한 설정
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))

    # 온도 예측 비교 그래프
    ax[0].scatter(actual_temp, predicted_temp, c='gold', edgecolor='black', alpha=0.75)
    ax[0].plot([actual_temp.min(), actual_temp.max()], [actual_temp.min(), actual_temp.max()], 'k--', lw=2)  # Identity line
    ax[0].set_xlabel('Actual Temperature')
    ax[0].set_ylabel('Predicted Temperature')
    ax[0].set_title('Temperature Prediction Comparison')
    ax[0].grid(True)

    # 습도 예측 비교 그래프
    ax[1].scatter(actual_humi, predicted_humi, c='gold', edgecolor='black', alpha=0.75)
    ax[1].plot([actual_humi.min(), actual_humi.max()], [actual_humi.min(), actual_humi.max()], 'k--', lw=2)  # Identity line
    ax[1].set_xlabel('Actual Humidity')
    ax[1].set_ylabel('Predicted Humidity')
    ax[1].set_title('Humidity Prediction Comparison')
    ax[1].grid(True)

    # 그래프 출력
    plt.tight_layout()
    plt.show()
# 결과 출력
for device_sn, result in results.items():
    print(f"Device {device_sn}: {result}")


# 암모니아 농도 예측 모델 생성 코드

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import os

# 데이터 로드 및 전처리
data = pd.read_excel('/content/drive/MyDrive/nh3 예측 폴더/data/selected_columns_0729min.xlsx')
data['timestamp'] = pd.to_datetime(data['fcst_datetime'])
data['hour'] = data['timestamp'].dt.hour
data['minute'] = data['timestamp'].dt.minute
data['dayofweek'] = data['timestamp'].dt.dayofweek

# 모델 저장 디렉토리 설정
model_dir = '/content/drive/MyDrive/nh3 예측 폴더/모델폴더'
os.makedirs(model_dir, exist_ok=True)

results = {}

# device_sn별로 데이터 분리
device_data = data.groupby('device_sn')
predicted_vs_actual = []  # 모든 예측 결과를 저장할 리스트

for device_sn, group in device_data:
    print(f"Processing {device_sn}")
    features = group[['temp', 'humi', 'tmp', 'wsd', 'reh', 'dayofweek', 'hour', 'minute']]
    targets = group['nh3_cv']
    timestamps = group['timestamp']  # timestamp 저장

    # 스케일링
    scaler_features = MinMaxScaler()
    scaler_targets = MinMaxScaler()
    features_scaled = scaler_features.fit_transform(features)
    targets_scaled = scaler_targets.fit_transform(targets.values.reshape(-1, 1))

    # 데이터 분할
    X_train, X_test, y_train, y_test, timestamps_train, timestamps_test = train_test_split(
        features_scaled, targets_scaled, timestamps, test_size=0.2, random_state=42)

    # 랜덤 포레스트 모델 훈련
    rf_model = RandomForestRegressor(max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=600, random_state=42)
    rf_model.fit(X_train, y_train.ravel())

    # 모델 저장
    model_filename = os.path.join(model_dir, f'rf_model_0729_min_{device_sn}.pkl')
    scaler_features_filename = os.path.join(model_dir, f'scaler_features_0729_min_{device_sn}.pkl')
    scaler_targets_filename = os.path.join(model_dir, f'scaler_targets_0729_min_{device_sn}.pkl')

    joblib.dump(rf_model, model_filename)
    joblib.dump(scaler_features, scaler_features_filename)
    joblib.dump(scaler_targets, scaler_targets_filename)

    # 예측 및 성능 평가
    predicted_nh3_rf = rf_model.predict(X_test)
    predicted_nh3_rf = scaler_targets.inverse_transform(predicted_nh3_rf.reshape(-1, 1))
    y_test_rescaled_rf = scaler_targets.inverse_transform(y_test.reshape(-1, 1))

    mse_rf = mean_squared_error(y_test_rescaled_rf, predicted_nh3_rf)
    rmse_rf = np.sqrt(mse_rf)
    r2_rf = r2_score(y_test_rescaled_rf, predicted_nh3_rf)
    mae_rf = mean_absolute_error(y_test_rescaled_rf, predicted_nh3_rf)

    print(f'[{device_sn}] RF NH3 MSE: {mse_rf}, RMSE: {rmse_rf}, R2: {r2_rf}, MAE: {mae_rf}')

    # 결과 저장
    results[device_sn] = {
        'mse': mse_rf,
        'rmse': rmse_rf,
        'r2': r2_rf,
        'mae': mae_rf,
        'predicted': predicted_nh3_rf,
        'actual': y_test_rescaled_rf
    }

    # 원래의 특성값들 중 테스트 셋에 해당하는 값들을 복원
    X_test_original = scaler_features.inverse_transform(X_test)

    # 실제값과 예측값을 데이터프레임 형태로 저장
    df_temp = pd.DataFrame({
        'Device_SN': device_sn,
        'Timestamp': timestamps_test.reset_index(drop=True),
        'temp': X_test_original[:, 0],
        'humi': X_test_original[:, 1],
        'tmp': X_test_original[:, 2],
        'wsd': X_test_original[:, 3],
        'reh': X_test_original[:, 4],
        'dayofweek': X_test_original[:, 5],
        'hour': X_test_original[:, 6],
        'Actual NH3': y_test_rescaled_rf.flatten(),
        'Predicted NH3': predicted_nh3_rf.flatten(),
    })
    predicted_vs_actual.append(df_temp)

     # 결과 시각화
    plt.figure(figsize=(10, 6))
    plt.scatter(range(len(y_test_rescaled_rf)), y_test_rescaled_rf, color='blue', label='Actual NH3 Concentration')
    plt.scatter(range(len(predicted_nh3_rf)), predicted_nh3_rf, color='red', label='Predicted NH3 Concentration')
    plt.title(f'Comparison of Actual and Predicted NH3 Concentration for Device {device_sn} using Random Forest')
    plt.xlabel('Test Sample Index')
    plt.ylabel('NH3 Concentration')
    plt.legend()
    plt.show()

# 각 장치별 정확도 계산 및 출력
threshold = 0.01

for device_sn, result in results.items():
    errors = np.abs(result['predicted'].flatten() - result['actual'].flatten())  # 배열을 1차원으로 평탄화
    accurate_predictions = np.sum(errors < threshold)
    accuracy_rate = (accurate_predictions / len(result['predicted'])) * 100

    print(f"[{device_sn}] 실제 NH3 농도의 총 개수: {len(result['predicted'])}, 정확한 예측 수: {accurate_predictions}, 정확도: {accuracy_rate:.2f}%")

# 모든 장치의 예측값과 실제값을 하나의 데이터프레임으로 결합
predicted_vs_actual_df = pd.concat(predicted_vs_actual, ignore_index=True)

# 결과 출력
display(predicted_vs_actual_df)

# 모델 불러와서 예측하는 코드

In [ ]:
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# 데이터 로드
data = pd.read_excel('/content/drive/MyDrive/nh3 예측 폴더/data/selected_columns_0729min.xlsx')
data['timestamp'] = pd.to_datetime(data['fcst_datetime'])
data['hour'] = data['timestamp'].dt.hour
data['minute'] = data['timestamp'].dt.minute
data['dayofweek'] = data['timestamp'].dt.dayofweek
# device_sn별로 데이터를 분리
device_groups = data.groupby('device_sn')

# 결과를 저장할 데이터프레임 초기화
results_df = pd.DataFrame()

# 각 device_sn별로 처리
for device_sn, group in device_groups:
    print(f"Processing device: {device_sn}")

    # 모델과 스케일러 로드
    model_temp = joblib.load(f'/content/drive/MyDrive/nh3 예측 폴더/model_temp_0729_min_{device_sn}.pkl')
    model_humi = joblib.load(f'/content/drive/MyDrive/nh3 예측 폴더/model_humi_0729_min_{device_sn}.pkl')
    model_nh3 = joblib.load(f'/content/drive/MyDrive/nh3 예측 폴더/모델폴더/rf_model_0729_min_{device_sn}.pkl')
    scaler_features = joblib.load(f'/content/drive/MyDrive/nh3 예측 폴더/모델폴더/scaler_features_0729_min_{device_sn}.pkl')
    scaler_targets = joblib.load(f'/content/drive/MyDrive/nh3 예측 폴더/모델폴더/scaler_targets_0729_min_{device_sn}.pkl')

    # 온도 및 습도 예측
    predicted_temp = model_temp.predict(group[['tmp', 'reh', 'pcp', 'wsd', 'sky','hour','minute']])
    predicted_humi = model_humi.predict(group[['tmp', 'reh', 'pcp', 'wsd', 'sky','hour','minute']])

    # 암모니아 농도 예측을 위한 특성 조합
    features_ammonia = np.column_stack((predicted_temp, predicted_humi, group['tmp'], group['wsd'], group['reh'], group['dayofweek'], group['hour'], group['minute']))
    features_ammonia_scaled = scaler_features.transform(features_ammonia)
    # X_ammonia_scaled = features_ammonia_scaled.reshape((features_ammonia_scaled.shape[0], 1, features_ammonia_scaled.shape[1]))
    X_ammonia_scaled = features_ammonia_scaled.reshape((features_ammonia_scaled.shape[0], -1))

    # 암모니아 농도 예측 및 스케일 되돌리기
    predicted_nh3_scaled = model_nh3.predict(X_ammonia_scaled)
    predicted_nh3 = scaler_targets.inverse_transform(predicted_nh3_scaled.reshape(-1, 1))
    actual_nh3 = group['nh3_cv'].values

     # 데이터 수집
    temp_df = pd.DataFrame({
        'device_sn': device_sn,
        'temp': group['tmp'],
        'humidity': group['reh'],
        'precipitation': group['pcp'],
        'wind_speed': group['wsd'],
        'sky': group['sky'],
        'hour': group['hour'],
        'minute': group['minute'],
        'predicted_temp': predicted_temp.flatten(),
        'predicted_humidity': predicted_humi.flatten(),
        'predicted_nh3': predicted_nh3.flatten(),
        'actual_nh3': actual_nh3,
        'prediction_time' : group['timestamp']
    })
    results_df = pd.concat([results_df, temp_df], ignore_index=True)

    # 결과 출력
    print(results_df)

    # 정확도 계산
    threshold = 0.01
    errors = np.abs(predicted_nh3.flatten() - actual_nh3)
    accurate_predictions = np.sum(errors < threshold)
    accuracy_rate = (accurate_predictions / len(predicted_nh3)) * 100

    print(f"[{device_sn}] NH3 MSE: {mean_squared_error(actual_nh3, predicted_nh3)}, R2: {r2_score(actual_nh3, predicted_nh3)}")
    print(f"[{device_sn}] 정확한 예측 수: {accurate_predictions}, 정확도: {accuracy_rate:.2f}%")

    # 결과 시각화
    plt.figure(figsize=(10, 6))
    plt.scatter(range(len(actual_nh3)), actual_nh3, color='blue', label='Actual NH3 Concentration')
    plt.scatter(range(len(predicted_nh3)), predicted_nh3, color='red', label='Predicted NH3 Concentration')
    plt.title(f'Comparison of Actual and Predicted NH3 Concentration for Device {device_sn}')
    plt.xlabel('Test Sample Index')
    plt.ylabel('NH3 Concentration')
    plt.legend()
    plt.show()
